# V16.1 — NO_WRITE
70 real V15 rows; no secrets, HTTP, SQL or scheduler.

In [ ]:
from pathlib import Path
import hashlib, json, runpy, socket, sys, zipfile

# Network and credentials are not needed or used in this notebook.
def deny_network(*args, **kwargs):
    raise RuntimeError("NO_WRITE notebook forbids network access")
socket.socket.connect = deny_network
socket.create_connection = deny_network
archives = list(Path('/kaggle/input').rglob('v161-no-write.zip'))
assert len(archives) == 1
root = Path('/kaggle/working/v161')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        target = (root/member.filename).resolve()
        assert root.resolve() in target.parents
    archive.extractall(root)
manifest = json.loads((root/'sha256_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == expected, name
sys.path.insert(0,str(root/'src'))
inputs = root/'inputs'
output = Path('/kaggle/working/v16_migration/baseline-v15-live')
sys.argv = ['30_migrate_v16.py','--input',str(inputs/'v15_canonical_rows.json'),
    '--notion-schema',str(inputs/'current_notion_schema.json'),
    '--live-review-snapshot',str(inputs/'live_knowledge_snapshot.json'),
    '--output-dir',str(output),'--run-id','V16.1-BASELINE-NO-WRITE','--expected-count','70']
runpy.run_path(str(root/'notebooks/30_migrate_v16.py'),run_name='__main__')
sys.argv = ['verify','--root',str(root),'--inputs',str(inputs),'--output',str(output)]
runpy.run_path(str(root/'scripts/verify_v161_no_write.py'),run_name='__main__')
print((output/'prepublication_validation_report.json').read_text())
